# Model C: character CNN

Trains on the same characters as models A and B: same `char_per_class`, seed and held-out tenth.
Needs the project on Drive at `MyDrive/ocr-project` and `data/rendered_<script>.zip` from
`render_chars.ipynb`. Everything is unzipped to the Colab disk and read from there. Saves
`models/cnn_<script>.pth` and `.json`; download both into `models/`. Runtime: T4 GPU.

In [ ]:
# Colab only
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/ocr-project
!pip -q install jiwer

In [ ]:
import os
import sys
sys.path.insert(0, 'src')  # first on the path, so our datasets.py wins over the Hugging Face package
import torch
from torch.utils.data import DataLoader
from classical import char_split, fingerprint
from datasets import CharDataset, class_list, load_charset, load_config
from models import CharCNN, cnn_step, fit, save_model, set_seed

SCRIPT = 'kannada'  # or 'english'
WORKERS = 2
cfg = load_config()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device, torch.cuda.get_device_name(0) if device == 'cuda' else '')

In [ ]:
# Colab only
!unzip -qo data/rendered_{SCRIPT}.zip -d /content/rendered
cfg['paths']['rendered'] = '/content/rendered'

In [ ]:
# Colab only: training reads from the Colab disk, so the optional Chars74K folders are copied there too
if os.path.isdir('data/public/chars74k'):
    !cp -r data/public/chars74k /content/
for key in ('chars74k_english', 'chars74k_kannada', 'chars74k_kannada_map'):
    cfg['public'][key] = cfg['public'][key].replace('data/public/chars74k', '/content/chars74k')

In [ ]:
set_seed(cfg['seed'])
train_set, val_set = char_split(cfg, SCRIPT, cfg['char_per_class'])
classes = class_list(load_charset(cfg['charsets'][SCRIPT]))
t = cfg['train']
train_loader = DataLoader(CharDataset(train_set, classes, cfg['char_size']), batch_size=t['batch_size'],
                          shuffle=True, num_workers=WORKERS)
val_loader = DataLoader(CharDataset(val_set, classes, cfg['char_size']), batch_size=t['batch_size'],
                        num_workers=WORKERS)

The fingerprint printed above should match the one `classical.py` printed for this script. If it does, A, B and C saw the same characters.

In [ ]:
model = CharCNN(len(classes)).to(device)
step = lambda m, batch, decode: cnn_step(m, batch, device, classes, decode)
result = fit(model, step, train_loader, val_loader, t['lr'], t['max_epochs'], t['patience'])

In [ ]:
os.makedirs(cfg['paths']['models'], exist_ok=True)
path = os.path.join(cfg['paths']['models'], f'cnn_{SCRIPT}.pth')
info = dict(kind='cnn', script=SCRIPT, classes=classes, size=cfg['char_size'], seed=cfg['seed'],
            char_per_class=cfg['char_per_class'], n_train=len(train_set),
            fingerprint=fingerprint(val_set + train_set), **result)
save_model(model.cpu(), info, path)
n_params = sum(p.numel() for p in model.parameters())
print(f'saved {path}: {os.path.getsize(path) / 1e6:.1f} MB, {n_params:,} parameters')